### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
bks = spark.table('bookings')
mems = spark.table('members')
fac = spark.table('facilities')

In [0]:
# Assuming you have loaded your tables into DataFrames named 'bookings' and 'members'
result = (
    bks
    .join(
        mems,
        bks["memid"] == mems["memid"],
        "inner"
    )
    .filter(
        (mems["firstname"] == "David") & 
        (mems["surname"] == "Farrell")
    )
    .select("starttime")
)

# To see the results:
result.show()

+-------------------+
|          starttime|
+-------------------+
|2012-09-18 09:00:00|
|2012-09-18 17:30:00|
|2012-09-18 13:30:00|
|2012-09-18 20:00:00|
|2012-09-19 09:30:00|
|2012-09-19 15:00:00|
|2012-09-19 12:00:00|
|2012-09-20 15:30:00|
|2012-09-20 11:30:00|
|2012-09-20 14:00:00|
|2012-09-21 10:30:00|
|2012-09-21 14:00:00|
|2012-09-22 08:30:00|
|2012-09-22 17:00:00|
|2012-09-23 08:30:00|
|2012-09-23 17:30:00|
|2012-09-23 19:00:00|
|2012-09-24 08:00:00|
|2012-09-24 16:30:00|
|2012-09-24 12:30:00|
+-------------------+
only showing top 20 rows


### Question

How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

https://pgexercises.com/questions/joins/simplejoin2.html

In [0]:
result = (
    fac
    .join(
        bks,
        fac["facid"] == bks["facid"],
        "inner"
    )
    .filter(
        fac["name"].isin("Tennis Court 1", "Tennis Court 2") &
        (bks["starttime"] >= "2012-09-21") &
        (bks["starttime"] < "2012-09-22")
    )
    .select(
        bks["starttime"].alias("start"),
        fac["name"].alias("name")
    )
    .orderBy(bks["starttime"])
)

# To view the output:
result.show()

+-------------------+--------------+
|              start|          name|
+-------------------+--------------+
|2012-09-21 08:00:00|Tennis Court 2|
|2012-09-21 08:00:00|Tennis Court 1|
|2012-09-21 09:30:00|Tennis Court 1|
|2012-09-21 10:00:00|Tennis Court 2|
|2012-09-21 11:30:00|Tennis Court 2|
|2012-09-21 12:00:00|Tennis Court 1|
|2012-09-21 13:30:00|Tennis Court 1|
|2012-09-21 14:00:00|Tennis Court 2|
|2012-09-21 15:30:00|Tennis Court 1|
|2012-09-21 16:00:00|Tennis Court 2|
|2012-09-21 17:00:00|Tennis Court 1|
|2012-09-21 18:00:00|Tennis Court 2|
+-------------------+--------------+



### Question

How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).

https://pgexercises.com/questions/joins/self2.html

In [0]:
# Use .alias() instead of .copy() to give Spark distinct names
mems_df = mems.alias("mems")
recs_df = mems.alias("recs")

result = (
    mems_df.join(
        recs_df,
        recs_df["memid"] == mems_df["recommendedby"],
        "left"
    )
    .select(
        mems_df["firstname"].alias("memfname"),
        mems_df["surname"].alias("memsname"),
        recs_df["firstname"].alias("recfname"),
        recs_df["surname"].alias("recsname")
    )
    .orderBy("memsname", "memfname")
)

result.show()

+---------+---------+---------+--------+
| memfname| memsname| recfname|recsname|
+---------+---------+---------+--------+
| Florence|    Bader|   Ponder|Stibbons|
|     Anne|    Baker|   Ponder|Stibbons|
|  Timothy|    Baker|   Jemima| Farrell|
|      Tim|   Boothe|      Tim|  Rownam|
|   Gerald|  Butters|   Darren|   Smith|
|     Joan|   Coplin|  Timothy|   Baker|
|    Erica|  Crumpet|    Tracy|   Smith|
|    Nancy|     Dare|   Janice|Joplette|
|    David|  Farrell|     NULL|    NULL|
|   Jemima|  Farrell|     NULL|    NULL|
|    GUEST|    GUEST|     NULL|    NULL|
|  Matthew|  Genting|   Gerald| Butters|
|     John|     Hunt|Millicent| Purview|
|    David|    Jones|   Janice|Joplette|
|  Douglas|    Jones|    David|   Jones|
|   Janice| Joplette|   Darren|   Smith|
|     Anna|Mackenzie|   Darren|   Smith|
|  Charles|     Owen|   Darren|   Smith|
|    David|   Pinker|   Jemima| Farrell|
|Millicent|  Purview|    Tracy|   Smith|
+---------+---------+---------+--------+
only showing top

### Question

How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

https://pgexercises.com/questions/joins/threejoin.html

In [0]:
from pyspark.sql import functions as F

result = (
    mems.alias("mems")
    .join(
        bks.alias("bks"), 
        mems["memid"] == bks["memid"], 
        "inner"
    )
    .join(
        fac.alias("facs"), 
        bks["facid"] == fac["facid"], 
        "inner"
    )
    .filter(
        fac["name"].isin("Tennis Court 1", "Tennis Court 2")
    )
    .select(
        F.concat_ws(" ", mems["firstname"], mems["surname"]).alias("member"),
        fac["name"].alias("facility")
    )
    .distinct()
    .orderBy("member", "facility")
)

# To view the output:
result.show()

+--------------+--------------+
|        member|      facility|
+--------------+--------------+
|    Anne Baker|Tennis Court 1|
|    Anne Baker|Tennis Court 2|
|  Burton Tracy|Tennis Court 1|
|  Burton Tracy|Tennis Court 2|
|  Charles Owen|Tennis Court 1|
|  Charles Owen|Tennis Court 2|
|  Darren Smith|Tennis Court 2|
| David Farrell|Tennis Court 1|
| David Farrell|Tennis Court 2|
|   David Jones|Tennis Court 1|
|   David Jones|Tennis Court 2|
|  David Pinker|Tennis Court 1|
| Douglas Jones|Tennis Court 1|
| Erica Crumpet|Tennis Court 1|
|Florence Bader|Tennis Court 1|
|Florence Bader|Tennis Court 2|
|   GUEST GUEST|Tennis Court 1|
|   GUEST GUEST|Tennis Court 2|
|Gerald Butters|Tennis Court 1|
|Gerald Butters|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows


### Question

How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.

https://pgexercises.com/questions/joins/sub.html

In [0]:
from pyspark.sql import functions as F

# Create distinct alias references for the self-join
mems_df = mems.alias("mems")
recs_df = mems.alias("recs")

result = (
    mems_df.join(
        recs_df,
        recs_df["memid"] == mems_df["recommendedby"],
        "left"
    )
    .select(
        F.concat_ws(" ", mems_df["firstname"], mems_df["surname"]).alias("member"),
        F.concat_ws(" ", recs_df["firstname"], recs_df["surname"]).alias("recommender")
    )
    .distinct()
    .orderBy("member")
)

# To view the output:
result.show()

+--------------------+---------------+
|              member|    recommender|
+--------------------+---------------+
|      Anna Mackenzie|   Darren Smith|
|          Anne Baker|Ponder Stibbons|
|        Burton Tracy|               |
|        Charles Owen|   Darren Smith|
|        Darren Smith|               |
|       David Farrell|               |
|         David Jones|Janice Joplette|
|        David Pinker| Jemima Farrell|
|       Douglas Jones|    David Jones|
|       Erica Crumpet|    Tracy Smith|
|      Florence Bader|Ponder Stibbons|
|         GUEST GUEST|               |
|      Gerald Butters|   Darren Smith|
|    Henrietta Rumney|Matthew Genting|
|Henry Worthington...|    Tracy Smith|
| Hyacinth Tupperware|               |
|          Jack Smith|   Darren Smith|
|     Janice Joplette|   Darren Smith|
|      Jemima Farrell|               |
|         Joan Coplin|  Timothy Baker|
+--------------------+---------------+
only showing top 20 rows


### Question

Produce a count of the number of recommendations each member has made. Order by member ID.

https://pgexercises.com/questions/aggregates/count3.html

In [0]:
result = (
    mems.filter(mems["recommendedby"].isNotNull())
    .groupBy("recommendedby")
    .agg(F.count("*").alias("count"))
    .orderBy("recommendedby")
)

# To view the output:
result.show()

+-------------+-----+
|recommendedby|count|
+-------------+-----+
|            1|    5|
|            2|    3|
|            3|    1|
|            4|    2|
|            5|    1|
|            6|    1|
|            9|    2|
|           11|    1|
|           13|    2|
|           15|    1|
|           16|    1|
|           20|    1|
|           30|    1|
+-------------+-----+



### Question

Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.

https://pgexercises.com/questions/aggregates/fachours.html

In [0]:
result = (
    bks.groupBy("facid")
    .agg(F.sum("slots").alias("Total Slots"))
    .orderBy("facid")
)

# To view the output:
result.show()

+-----+-----------+
|facid|Total Slots|
+-----+-----------+
|    0|       1320|
|    1|       1278|
|    2|       1209|
|    3|        830|
|    4|       1404|
|    5|        228|
|    6|       1104|
|    7|        908|
|    8|        911|
+-----+-----------+



### Question

Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

https://pgexercises.com/questions/aggregates/fachoursbymonth.html

In [0]:
result = (
    bks.filter(
        (bks["starttime"] >= "2012-09-01") & 
        (bks["starttime"] < "2012-10-01")
    )
    .groupBy("facid")
    .agg(F.sum("slots").alias("Total Slots"))
    .orderBy("Total Slots")
)

# To view the output:
result.show()

+-----+-----------+
|facid|Total Slots|
+-----+-----------+
|    5|        122|
|    3|        422|
|    7|        426|
|    8|        471|
|    6|        540|
|    2|        570|
|    1|        588|
|    0|        591|
|    4|        648|
+-----+-----------+



### Question

Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

https://pgexercises.com/questions/aggregates/fachoursbymonth2.html

In [0]:
result = (
    bks.filter(F.year(bks["starttime"]) == 2012)
    .withColumn("month", F.month(bks["starttime"]))
    .groupBy("facid", "month")
    .agg(F.sum("slots").alias("Total Slots"))
    .orderBy("facid", "month")
)

# To view the output:
result.show()

+-----+-----+-----------+
|facid|month|Total Slots|
+-----+-----+-----------+
|    0|    7|        270|
|    0|    8|        459|
|    0|    9|        591|
|    1|    7|        207|
|    1|    8|        483|
|    1|    9|        588|
|    2|    7|        180|
|    2|    8|        459|
|    2|    9|        570|
|    3|    7|        104|
|    3|    8|        304|
|    3|    9|        422|
|    4|    7|        264|
|    4|    8|        492|
|    4|    9|        648|
|    5|    7|         24|
|    5|    8|         82|
|    5|    9|        122|
|    6|    7|        164|
|    6|    8|        400|
+-----+-----+-----------+
only showing top 20 rows


### Question

Find the total number of members (including guests) who have made at least one booking.

https://pgexercises.com/questions/aggregates/members1.html

In [0]:
result = bks.select(F.count_distinct(bks["memid"]).alias("count"))

# To view the output:
result.show()

+-----+
|count|
+-----+
|   30|
+-----+



### Question

Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.

https://pgexercises.com/questions/aggregates/nbooking.html

In [0]:
result = (
    bks.alias("bks")
    .join(
        mems.alias("mems"),
        bks["memid"] == mems["memid"],
        "inner"
    )
    .filter(bks["starttime"] >= "2012-09-01")
    .groupBy(
        mems["surname"], 
        mems["firstname"], 
        mems["memid"]
    )
    .agg(F.min(bks["starttime"]).alias("starttime"))
    .orderBy(mems["memid"])
)

# To view the output:
result.show()

+---------+---------+-----+-------------------+
|  surname|firstname|memid|          starttime|
+---------+---------+-----+-------------------+
|    GUEST|    GUEST|    0|2012-09-01 08:00:00|
|    Smith|   Darren|    1|2012-09-01 09:00:00|
|    Smith|    Tracy|    2|2012-09-01 11:30:00|
|   Rownam|      Tim|    3|2012-09-01 16:00:00|
| Joplette|   Janice|    4|2012-09-01 15:00:00|
|  Butters|   Gerald|    5|2012-09-02 12:30:00|
|    Tracy|   Burton|    6|2012-09-01 15:00:00|
|     Dare|    Nancy|    7|2012-09-01 12:30:00|
|   Boothe|      Tim|    8|2012-09-01 08:30:00|
| Stibbons|   Ponder|    9|2012-09-01 11:00:00|
|     Owen|  Charles|   10|2012-09-01 11:00:00|
|    Jones|    David|   11|2012-09-01 09:30:00|
|    Baker|     Anne|   12|2012-09-01 14:30:00|
|  Farrell|   Jemima|   13|2012-09-01 09:30:00|
|    Smith|     Jack|   14|2012-09-01 11:00:00|
|    Bader| Florence|   15|2012-09-01 10:30:00|
|    Baker|  Timothy|   16|2012-09-01 15:00:00|
|   Pinker|    David|   17|2012-09-01 08

### Question

Output the names of all members, formatted as 'Surname, Firstname'

https://pgexercises.com/questions/string/concat.html

In [0]:
result = mems.select(
    F.concat_ws(", ", mems["surname"], mems["firstname"]).alias("name")
)

# To view the output:
result.show()

+----------------+
|            name|
+----------------+
|    GUEST, GUEST|
|   Smith, Darren|
|    Smith, Tracy|
|     Rownam, Tim|
|Joplette, Janice|
| Butters, Gerald|
|   Tracy, Burton|
|     Dare, Nancy|
|     Boothe, Tim|
|Stibbons, Ponder|
|   Owen, Charles|
|    Jones, David|
|     Baker, Anne|
| Farrell, Jemima|
|     Smith, Jack|
| Bader, Florence|
|  Baker, Timothy|
|   Pinker, David|
|Genting, Matthew|
| Mackenzie, Anna|
+----------------+
only showing top 20 rows


### Question

Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.

https://pgexercises.com/questions/string/case.html

In [0]:
result = fac.filter(
    F.upper(fac["name"]).like("TENNIS%")
)

# To view the output:
result.show()

+-----+--------------+----------+---------+-------------+------------------+
|facid|          name|membercost|guestcost|initiallayout|monthlymaintenance|
+-----+--------------+----------+---------+-------------+------------------+
|    0|Tennis Court 1|         5|       25|        10000|               200|
|    1|Tennis Court 2|         5|       25|         8000|               200|
+-----+--------------+----------+---------+-------------+------------------+



### Question

You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.

https://pgexercises.com/questions/string/reg.html

In [0]:
result = (
    mems.filter(mems["telephone"].rlike("[()]"))
    .select(mems["memid"], mems["telephone"])
)

# To view the output:
result.show()

+-----+--------------+
|memid|     telephone|
+-----+--------------+
|    0|(000) 000-0000|
|    3|(844) 693-0723|
|    4|(833) 942-4710|
|    5|(844) 078-4130|
|    6|(822) 354-9973|
|    7|(833) 776-4001|
|    8|(811) 433-2547|
|    9|(833) 160-3900|
|   10|(855) 542-5251|
|   11|(844) 536-8036|
|   13|(855) 016-0163|
|   14|(822) 163-3254|
|   15|(833) 499-3527|
|   20|(811) 972-1377|
|   21|(822) 661-2898|
|   22|(822) 499-2232|
|   24|(822) 413-1470|
|   27|(822) 989-8876|
|   28|(855) 755-9876|
|   29|(855) 894-3758|
+-----+--------------+
only showing top 20 rows


### Question

You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.

https://pgexercises.com/questions/string/substr.html

In [0]:
result = (
    mems.withColumn("letter", F.substring(mems["surname"], 1, 1))
    .groupBy("letter")
    .agg(F.count("*").alias("count"))
    .orderBy("letter")
)

# To view the output:
result.show()

+------+-----+
|letter|count|
+------+-----+
|     B|    5|
|     C|    2|
|     D|    1|
|     F|    2|
|     G|    2|
|     H|    1|
|     J|    3|
|     M|    1|
|     O|    1|
|     P|    2|
|     R|    2|
|     S|    6|
|     T|    2|
|     W|    1|
+------+-----+



### Question

Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.

https://pgexercises.com/questions/date/series.html

In [0]:
result = (
    spark.createDataFrame([(1,)], schema="id INT") # Creates a 1-row dummy DataFrame
    .select(
        F.explode(
            F.sequence(
                F.to_timestamp(F.lit("2012-10-01")), 
                F.to_timestamp(F.lit("2012-10-31")), 
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("ts")
    )
)

# To view the output:
result.show()

+-------------------+
|                 ts|
+-------------------+
|2012-10-01 00:00:00|
|2012-10-02 00:00:00|
|2012-10-03 00:00:00|
|2012-10-04 00:00:00|
|2012-10-05 00:00:00|
|2012-10-06 00:00:00|
|2012-10-07 00:00:00|
|2012-10-08 00:00:00|
|2012-10-09 00:00:00|
|2012-10-10 00:00:00|
|2012-10-11 00:00:00|
|2012-10-12 00:00:00|
|2012-10-13 00:00:00|
|2012-10-14 00:00:00|
|2012-10-15 00:00:00|
|2012-10-16 00:00:00|
|2012-10-17 00:00:00|
|2012-10-18 00:00:00|
|2012-10-19 00:00:00|
|2012-10-20 00:00:00|
+-------------------+
only showing top 20 rows


### Question

Return a count of bookings for each month, sorted by month

https://pgexercises.com/questions/date/bookingspermonth.html

In [0]:
result = (
    bks.withColumn("month", F.date_trunc("month", bks["starttime"]))
    .groupBy("month")
    .agg(F.count("*").alias("count"))
    .orderBy("month")
)

# To view the output:
result.show()

+-------------------+-----+
|              month|count|
+-------------------+-----+
|2012-07-01 00:00:00|  658|
|2012-08-01 00:00:00| 1472|
|2012-09-01 00:00:00| 1913|
|2013-01-01 00:00:00|    1|
+-------------------+-----+

